# Module 6 — Edge AI & TinyML: Deploying SLMs Beyond the Cloud
### Export Pipeline: MediLearn Fine-Tuned SmolLM → GGUF → Edge-Ready Artifact

> **What this notebook does:** Loads the **fine-tuned MediLearn model from Module 2**
> (saved to Google Drive), converts it to GGUF format, and quantizes it to Q4_K_M —
> a portable, CPU-first binary that runs locally with llama.cpp on any laptop.
>
> The final section re-runs the shared 60-question MediLearn evaluation against the
> quantized model and compares accuracy and latency to the full-precision Module 2 result,
> closing the workshop loop: every module has now scored the same 60 questions,
> and you can see — with one consistent chart — what each technique added.

---
**Workshop handoff chain:**
```
Module 1 → baseline score (base SmolLM, no fine-tuning)
Module 2 → fine-tuned score + merged model saved to Google Drive
Module 3 → fine-tuned SLM + RAG score
Module 4 → passive RAG vs agentic RAG score
Module 6 → full-precision vs Q4_K_M quantized score  ← you are here
```


---
## Section 1 — Environment Check


In [1]:
import platform, psutil, os, shutil

print('=== Runtime ===')
print(f'Python : {platform.python_version()}')
print(f'OS     : {platform.system()} {platform.machine()}')

ram_gb    = psutil.virtual_memory().total     / 1024**3
ram_avail = psutil.virtual_memory().available / 1024**3
print(f'RAM    : {ram_gb:.1f} GB total  |  {ram_avail:.1f} GB available')

disk = shutil.disk_usage('/content')
disk_free = disk.free / 1024**3
print(f'Disk   : {disk_free:.1f} GB free in /content')

try:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None (CPU-only — expected)'
    print(f'GPU    : {gpu}')
except ImportError:
    print('GPU    : torch not installed yet')

print()
if disk_free < 5.0:
    print('⚠️  Less than 5 GB free — may run out of space during conversion.')
    print('   Go to Runtime → Disconnect and delete runtime, then reconnect.')
else:
    print('✅ Disk space looks good.')

if ram_avail < 4.0:
    print('⚠️  Low RAM — if a step fails with OOM, restart runtime and try again.')
else:
    print('✅ RAM looks good.')

=== Runtime ===
Python : 3.12.13
OS     : Linux x86_64
RAM    : 12.7 GB total  |  11.5 GB available
Disk   : 65.3 GB free in /content
GPU    : Tesla T4

✅ Disk space looks good.
✅ RAM looks good.


---
## Section 2 — Install Dependencies


In [2]:
%%capture
import subprocess, sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface_hub',
    'transformers>=4.40.0',
    'gguf',
    'sentencepiece',
    'psutil',
])
print('✅ Done')

In [3]:
import huggingface_hub, transformers
print(f'huggingface_hub : {huggingface_hub.__version__}')
print(f'transformers    : {transformers.__version__}')
print('✅ All imports OK')

huggingface_hub : 1.11.0
transformers    : 5.0.0
✅ All imports OK


---
## Section 2.5 — Mount Google Drive & Load Shared Evaluation Set

The fine-tuned model lives on Drive (produced by Module 2). We also load the shared
60-question MediLearn evaluation set — same set, same scoring code as every other module.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

WORKSHOP_DIR     = "/content/drive/MyDrive/SLM Workshop"
EVAL_DIR         = f"{WORKSHOP_DIR}/eval"
DRIVE_RESULTS_DIR = f"{EVAL_DIR}/results"
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

# Canonical path Module 2's Section 8.1 writes to
MERGED_MODEL_PATH = f"{WORKSHOP_DIR}/medical_slm_finetuned"

if not os.path.exists(MERGED_MODEL_PATH):
    raise FileNotFoundError(
        f"\n=================================================================\n"
        f"Fine-tuned MediLearn model not found at:\n  {MERGED_MODEL_PATH}\n\n"
        f"Run Module 2 through Section 8.1 (Merge Adapter & Save to Drive) first.\n"
        f"Do NOT substitute a different model — this notebook's eval comparison\n"
        f"is only meaningful if all modules use the same fine-tuned weights.\n"
        f"================================================================="
    )
print(f"✅ Fine-tuned model found at: {MERGED_MODEL_PATH}")

# Shared eval set
MEDILEARN_EVAL_JSON = '''[{"id":"Q001","category":"Cardiology","difficulty":"L1","question":"What is the normal resting heart rate range for a healthy adult?","key_terms":["60","100","beats per minute","bpm"],"reference_answer":"A normal resting heart rate for adults is approximately 60-100 beats per minute."},{"id":"Q002","category":"Cardiology","difficulty":"L1","question":"What does the P wave represent on an ECG?","key_terms":["atrial","depolarization","p wave"],"reference_answer":"The P wave represents atrial depolarization, the electrical activation of the atria before they contract."},{"id":"Q003","category":"Cardiology","difficulty":"L1","question":"What do the S1 and S2 heart sounds represent?","key_terms":["mitral","tricuspid","aortic","pulmonic","valve closure"],"reference_answer":"S1 is caused by closure of the mitral and tricuspid valves at the start of systole; S2 is caused by closure of the aortic and pulmonic valves at the start of diastole."},{"id":"Q004","category":"Cardiology","difficulty":"L2","question":"A 58-year-old smoker presents with substernal chest pain radiating to the left arm, diaphoresis, and dyspnea. What is the most likely diagnosis and first-line antiplatelet therapy?","key_terms":["myocardial infarction","mi","acute coronary syndrome","aspirin","clopidogrel","p2y12"],"reference_answer":"This presentation is consistent with acute myocardial infarction (ACS). First-line antiplatelet therapy is aspirin, typically combined with a P2Y12 inhibitor such as clopidogrel."},{"id":"Q005","category":"Cardiology","difficulty":"L2","question":"What ECG finding distinguishes an NSTEMI from a STEMI?","key_terms":["st elevation","st segment","troponin","nstemi","stemi"],"reference_answer":"STEMI shows persistent ST-segment elevation on ECG, while NSTEMI does not show ST elevation but troponin is still elevated, indicating myocardial injury without complete vessel occlusion."},{"id":"Q006","category":"Cardiology","difficulty":"L2","question":"A patient with COPD presents with dyspnea, orthopnea, JVD, and an S3 gallop. What is the likely diagnosis and first-line medication class?","key_terms":["heart failure","diuretic","loop diuretic","fluid overload"],"reference_answer":"These findings suggest decompensated heart failure with volume overload. First-line treatment is a loop diuretic such as furosemide to reduce fluid overload."},{"id":"Q007","category":"Cardiology","difficulty":"L2","question":"What is the recommended blood pressure target for a patient with hypertension and chronic kidney disease per current guidelines?","key_terms":["130","80","mmhg","target"],"reference_answer":"Current guidelines generally recommend a blood pressure target below 130/80 mmHg for patients with hypertension and chronic kidney disease."},{"id":"Q008","category":"Cardiology","difficulty":"L3","question":"A 72-year-old with HFrEF (EF 30%) remains symptomatic despite an ACE inhibitor and beta-blocker. What two additional guideline-directed therapies should be considered, and why?","key_terms":["mineralocorticoid receptor antagonist","mra","sglt2 inhibitor","spironolactone","mortality benefit"],"reference_answer":"Guideline-directed therapy adds a mineralocorticoid receptor antagonist (e.g., spironolactone) and an SGLT2 inhibitor, both of which have demonstrated mortality and hospitalization benefit on top of ACE inhibitor and beta-blocker therapy in HFrEF."},{"id":"Q009","category":"Cardiology","difficulty":"L3","question":"Explain why beta-blockers are avoided in acute decompensated heart failure with cardiogenic shock but used in chronic stable heart failure.","key_terms":["negative inotrope","cardiac output","cardiogenic shock","chronic remodeling"],"reference_answer":"In acute cardiogenic shock, beta-blockers further reduce contractility and cardiac output, worsening hypoperfusion. In chronic stable heart failure, beta-blockers reduce sympathetic overdrive and adverse remodeling over time, improving long-term survival once the patient is hemodynamically stable."},{"id":"Q010","category":"Cardiology","difficulty":"L3","question":"A patient on warfarin presents with an INR of 8 and no active bleeding. What is the appropriate management?","key_terms":["hold warfarin","vitamin k","oral vitamin k","inr"],"reference_answer":"Warfarin should be held, and oral vitamin K should be administered to reduce the INR, since the patient is not actively bleeding and rapid reversal with clotting factors is not required."},{"id":"Q011","category":"Neurology","difficulty":"L1","question":"Which cranial nerves are primarily responsible for eye movement?","key_terms":["cranial nerve iii","oculomotor","trochlear","abducens","cn iv","cn vi"],"reference_answer":"Eye movement is controlled by cranial nerve III (oculomotor), IV (trochlear), and VI (abducens)."},{"id":"Q012","category":"Neurology","difficulty":"L1","question":"What is the primary function of the cerebellum?","key_terms":["coordination","balance","motor","fine motor control"],"reference_answer":"The cerebellum coordinates voluntary movement, balance, and fine motor control."},{"id":"Q013","category":"Neurology","difficulty":"L1","question":"Define aphasia and name its two major types.","key_terms":["aphasia","broca","wernicke","language"],"reference_answer":"Aphasia is an impairment of language production or comprehension. The two major types are Broca's (expressive) aphasia and Wernicke's (receptive) aphasia."},{"id":"Q014","category":"Neurology","difficulty":"L2","question":"A patient presents with sudden facial drooping, arm weakness, and aphasia. What is the diagnostic and treatment window for thrombolysis?","key_terms":["stroke","tpa","thrombolysis","4.5 hours","ct"],"reference_answer":"This presentation suggests acute ischemic stroke. After a non-contrast CT rules out hemorrhage, IV tPA (thrombolysis) can be given within 4.5 hours of symptom onset."},{"id":"Q015","category":"Neurology","difficulty":"L2","question":"What CSF findings differentiate bacterial from viral meningitis?","key_terms":["glucose","protein","neutrophil","lymphocyte","csf"],"reference_answer":"Bacterial meningitis typically shows low glucose, high protein, and neutrophil predominance in CSF, while viral meningitis shows normal glucose, mildly elevated protein, and lymphocyte predominance."},{"id":"Q016","category":"Neurology","difficulty":"L2","question":"A patient with diplopia, ptosis, and symptoms that improve with rest is suspected to have which neuromuscular disorder, and what test confirms it?","key_terms":["myasthenia gravis","acetylcholine receptor antibody","edrophonium","ice pack test"],"reference_answer":"This presentation is consistent with myasthenia gravis. It can be confirmed with acetylcholine receptor antibody testing or the edrophonium (Tensilon) test."},{"id":"Q017","category":"Neurology","difficulty":"L2","question":"What is the first-line treatment for status epilepticus?","key_terms":["benzodiazepine","lorazepam","diazepam","iv"],"reference_answer":"First-line treatment for status epilepticus is an IV benzodiazepine, such as lorazepam or diazepam."},{"id":"Q018","category":"Neurology","difficulty":"L3","question":"A patient presents with ascending symmetric weakness after a recent gastrointestinal infection. What is the diagnosis, expected CSF finding, and key treatment?","key_terms":["guillain-barre","albuminocytologic dissociation","ivig","plasmapheresis"],"reference_answer":"This is consistent with Guillain-Barre syndrome. CSF typically shows albuminocytologic dissociation (elevated protein with normal cell count). Treatment is IVIG or plasmapheresis."},{"id":"Q019","category":"Neurology","difficulty":"L3","question":"A 65-year-old presents with a sudden severe headache described as 'the worst of my life,' with no focal deficits, and a negative CT. What is the next diagnostic step, and what condition must be ruled out?","key_terms":["lumbar puncture","subarachnoid hemorrhage","xanthochromia"],"reference_answer":"The next step is a lumbar puncture to look for xanthochromia, since a sudden severe ('thunderclap') headache raises concern for subarachnoid hemorrhage even with a negative CT."},{"id":"Q020","category":"Neurology","difficulty":"L3","question":"Compare the mechanism and clinical use of IV thrombolysis versus mechanical thrombectomy in acute ischemic stroke, including time windows.","key_terms":["tpa","thrombectomy","4.5 hours","24 hours","large vessel occlusion"],"reference_answer":"IV thrombolysis (tPA) dissolves the clot pharmacologically and is given within 4.5 hours of onset. Mechanical thrombectomy physically removes the clot via catheter and can be used up to 24 hours in select patients with large vessel occlusion, often combined with tPA."},{"id":"Q021","category":"Respiratory","difficulty":"L1","question":"What is the normal respiratory rate range for a resting adult?","key_terms":["12","20","breaths per minute"],"reference_answer":"A normal resting respiratory rate for adults is approximately 12-20 breaths per minute."},{"id":"Q022","category":"Respiratory","difficulty":"L1","question":"What does a low FEV1/FVC ratio indicate, obstructive or restrictive lung disease?","key_terms":["obstructive","fev1","fvc","ratio"],"reference_answer":"A low FEV1/FVC ratio indicates an obstructive pattern, as seen in conditions like asthma and COPD."},{"id":"Q023","category":"Respiratory","difficulty":"L1","question":"What is the most common cause of community-acquired pneumonia in healthy adults?","key_terms":["streptococcus pneumoniae","strep pneumoniae","pneumococcus"],"reference_answer":"Streptococcus pneumoniae is the most common causative organism of community-acquired pneumonia in healthy adults."},{"id":"Q024","category":"Respiratory","difficulty":"L2","question":"A 16-year-old presents with severe wheezing, SpO2 85%, and is unable to speak full sentences. What is the immediate management for this asthma exacerbation?","key_terms":["oxygen","albuterol","bronchodilator","corticosteroid","nebulized"],"reference_answer":"Immediate management includes supplemental oxygen, nebulized short-acting beta-agonist (albuterol) bronchodilators, and systemic corticosteroids."},{"id":"Q025","category":"Respiratory","difficulty":"L2","question":"A 72-year-old presents with productive cough, fever, and right lower lobe consolidation on chest X-ray. What is the recommended outpatient antibiotic approach for community-acquired pneumonia?","key_terms":["amoxicillin","macrolide","doxycycline","outpatient"],"reference_answer":"For outpatient community-acquired pneumonia, recommended regimens include amoxicillin, or a macrolide/doxycycline, depending on comorbidities and local resistance patterns."},{"id":"Q026","category":"Respiratory","difficulty":"L2","question":"What distinguishes Type 1 from Type 2 respiratory failure based on arterial blood gas findings?","key_terms":["hypoxemia","hypercapnia","pao2","paco2"],"reference_answer":"Type 1 respiratory failure is characterized by hypoxemia with normal or low PaCO2, while Type 2 respiratory failure involves hypoxemia with hypercapnia (elevated PaCO2)."},{"id":"Q027","category":"Respiratory","difficulty":"L2","question":"What is the mechanism by which COPD leads to chronic CO2 retention?","key_terms":["air trapping","alveolar destruction","ventilation perfusion mismatch","hypoventilation"],"reference_answer":"Airway obstruction and alveolar destruction in COPD cause air trapping and ventilation-perfusion mismatch, reducing effective alveolar ventilation and leading to chronic CO2 retention over time."},{"id":"Q028","category":"Respiratory","difficulty":"L3","question":"A patient with ARDS is being mechanically ventilated. What ventilation strategy is recommended to reduce mortality, and why?","key_terms":["low tidal volume","6 ml/kg","lung protective","barotrauma"],"reference_answer":"A low tidal volume (lung-protective) ventilation strategy of approximately 6 mL/kg of predicted body weight is recommended, as it reduces ventilator-induced lung injury and barotrauma, improving survival."},{"id":"Q029","category":"Respiratory","difficulty":"L3","question":"Explain why supplemental oxygen must be titrated carefully in chronic CO2 retainers.","key_terms":["hypoxic drive","co2 retention","respiratory drive"],"reference_answer":"In chronic CO2 retainers, the respiratory drive may become more dependent on hypoxemia (hypoxic drive) rather than CO2 levels; excessive supplemental oxygen can blunt this drive and worsen hypoventilation and CO2 retention."},{"id":"Q030","category":"Respiratory","difficulty":"L3","question":"A patient with suspected pulmonary embolism has a low pretest probability. What is the appropriate diagnostic pathway?","key_terms":["d-dimer","ct pulmonary angiography","wells score","pretest probability"],"reference_answer":"With a low pretest probability, a D-dimer test is appropriate first; if negative, PE can generally be excluded without further imaging. If positive or pretest probability is higher, CT pulmonary angiography is the next step."},{"id":"Q031","category":"Endocrine","difficulty":"L1","question":"What hormone is deficient in Type 1 diabetes mellitus?","key_terms":["insulin","beta cell"],"reference_answer":"Type 1 diabetes mellitus results from autoimmune destruction of pancreatic beta cells, causing an absolute deficiency of insulin."},{"id":"Q032","category":"Endocrine","difficulty":"L1","question":"What is the function of thyroid hormones T3 and T4?","key_terms":["metabolism","metabolic rate","thyroid"],"reference_answer":"T3 and T4 regulate the body's metabolic rate, influencing heart rate, temperature, and energy expenditure."},{"id":"Q033","category":"Endocrine","difficulty":"L1","question":"What does HbA1c reflect?","key_terms":["average blood glucose","3 months","glycated hemoglobin"],"reference_answer":"HbA1c reflects average blood glucose levels over the preceding 2-3 months, based on glycation of hemoglobin."},{"id":"Q034","category":"Endocrine","difficulty":"L2","question":"A 22-year-old presents with 3 weeks of polyuria, polydipsia, and weight loss. What is the likely diagnosis and the initial management?","key_terms":["type 1 diabetes","insulin","new-onset diabetes"],"reference_answer":"This presentation suggests new-onset Type 1 diabetes mellitus. Initial management is insulin therapy, with evaluation for diabetic ketoacidosis."},{"id":"Q035","category":"Endocrine","difficulty":"L2","question":"A patient on diuretics presents with weakness, muscle cramps, and palpitations; potassium is 2.8. How should hypokalemia be corrected?","key_terms":["potassium replacement","oral potassium","iv potassium","ecg monitoring"],"reference_answer":"Hypokalemia should be corrected with oral or IV potassium replacement, with cardiac (ECG) monitoring given the risk of arrhythmia, and the underlying cause (diuretic use) addressed."},{"id":"Q036","category":"Endocrine","difficulty":"L2","question":"What are the diagnostic criteria for diabetic ketoacidosis (DKA)?","key_terms":["hyperglycemia","ketosis","metabolic acidosis","anion gap"],"reference_answer":"DKA is diagnosed by the triad of hyperglycemia, ketosis (elevated ketones), and metabolic acidosis with an elevated anion gap."},{"id":"Q037","category":"Endocrine","difficulty":"L2","question":"What is the first-line pharmacologic treatment for newly diagnosed Type 2 diabetes mellitus?","key_terms":["metformin"],"reference_answer":"Metformin is the first-line pharmacologic treatment for newly diagnosed Type 2 diabetes mellitus, alongside lifestyle modification."},{"id":"Q038","category":"Endocrine","difficulty":"L3","question":"A patient in DKA is being treated with IV insulin. What electrolyte must be monitored closely and corrected before or during insulin therapy, and why?","key_terms":["potassium","hypokalemia","intracellular shift"],"reference_answer":"Potassium must be monitored closely. Insulin drives potassium intracellularly, which can precipitate severe hypokalemia and arrhythmia if potassium is not replaced before or alongside insulin therapy."},{"id":"Q039","category":"Endocrine","difficulty":"L3","question":"Explain the mechanism by which SGLT2 inhibitors lower blood glucose and their additional cardiorenal benefits.","key_terms":["sglt2","glucose reabsorption","glycosuria","cardiorenal","heart failure"],"reference_answer":"SGLT2 inhibitors block glucose reabsorption in the proximal renal tubule, causing glycosuria and lowering blood glucose. They also reduce intraglomerular pressure and have demonstrated cardiorenal benefits, including reduced heart failure hospitalizations and slowed progression of chronic kidney disease."},{"id":"Q040","category":"Endocrine","difficulty":"L3","question":"Compare the clinical presentation and biochemical findings of SIADH versus diabetes insipidus.","key_terms":["siadh","diabetes insipidus","hyponatremia","hypernatremia","urine osmolality"],"reference_answer":"SIADH causes water retention, hyponatremia, and concentrated urine due to excess ADH. Diabetes insipidus causes excessive dilute urine output and hypernatremia due to insufficient ADH action, either central (deficient ADH) or nephrogenic (renal resistance to ADH)."},{"id":"Q041","category":"Critical Care","difficulty":"L1","question":"What are the classic signs of meningeal irritation on physical exam?","key_terms":["nuchal rigidity","kernig","brudzinski","neck stiffness"],"reference_answer":"Classic signs of meningeal irritation include nuchal rigidity (neck stiffness), a positive Kernig's sign, and a positive Brudzinski's sign."},{"id":"Q042","category":"Critical Care","difficulty":"L1","question":"Define sepsis according to the Sepsis-3 criteria.","key_terms":["life-threatening","organ dysfunction","infection","dysregulated response"],"reference_answer":"Sepsis-3 defines sepsis as life-threatening organ dysfunction caused by a dysregulated host response to infection."},{"id":"Q043","category":"Critical Care","difficulty":"L1","question":"What is the empiric antibiotic of choice for suspected bacterial meningitis in an adult?","key_terms":["ceftriaxone","vancomycin","empiric"],"reference_answer":"Empiric therapy for suspected bacterial meningitis in adults typically includes ceftriaxone plus vancomycin, often with dexamethasone."},{"id":"Q044","category":"Critical Care","difficulty":"L2","question":"A 54-year-old presents with septic shock from urosepsis (fever 40C, BP 88/52, HR 128). What are the key components of the sepsis bundle, and within what time frame should they be initiated?","key_terms":["blood cultures","lactate","fluids","antibiotics","1 hour"],"reference_answer":"The sepsis bundle includes obtaining blood cultures, measuring lactate, starting broad-spectrum antibiotics, and beginning IV fluid resuscitation, ideally all within the first hour of recognition."},{"id":"Q045","category":"Critical Care","difficulty":"L2","question":"What CSF glucose and protein pattern is expected in bacterial versus viral meningitis?","key_terms":["low glucose","high protein","bacterial","viral","normal glucose"],"reference_answer":"Bacterial meningitis typically shows low CSF glucose and high protein, while viral meningitis typically shows normal glucose and only mildly elevated protein."},{"id":"Q046","category":"Critical Care","difficulty":"L2","question":"A patient with septic shock remains hypotensive despite adequate fluid resuscitation. What is the first-line vasopressor?","key_terms":["norepinephrine","vasopressor"],"reference_answer":"Norepinephrine is the first-line vasopressor for septic shock that persists despite adequate fluid resuscitation."},{"id":"Q047","category":"Critical Care","difficulty":"L2","question":"What is the recommended timing for antibiotic administration in suspected septic shock?","key_terms":["1 hour","as soon as possible","broad-spectrum"],"reference_answer":"Broad-spectrum antibiotics should be administered as soon as possible, ideally within 1 hour of recognizing septic shock, after blood cultures are drawn."},{"id":"Q048","category":"Critical Care","difficulty":"L3","question":"Explain why early antibiotic administration in septic shock improves survival.","key_terms":["mortality","delay","golden hour","organ dysfunction"],"reference_answer":"Each hour of delay in appropriate antibiotic therapy during septic shock is associated with increased mortality, as ongoing untreated infection drives progressive organ dysfunction and hemodynamic collapse; early treatment interrupts this cascade."},{"id":"Q049","category":"Critical Care","difficulty":"L3","question":"A neutropenic febrile cancer patient presents with fever. What is the recommended empiric antibiotic approach, and why does timing matter so much in this population?","key_terms":["broad-spectrum","pseudomonas","neutropenic fever","immediate"],"reference_answer":"Neutropenic fever requires immediate broad-spectrum antibiotics with antipseudomonal coverage (e.g., cefepime or piperacillin-tazobactam), because these patients lack a normal immune response and can deteriorate rapidly from untreated infection."},{"id":"Q050","category":"Critical Care","difficulty":"L3","question":"Discuss the rationale for adding dexamethasone in bacterial meningitis treatment.","key_terms":["dexamethasone","inflammation","streptococcus pneumoniae","neurological outcome"],"reference_answer":"Dexamethasone is given alongside antibiotics in bacterial meningitis, particularly when Streptococcus pneumoniae is suspected, to reduce the inflammatory response to bacterial lysis and improve neurological outcomes, including reducing hearing loss."},{"id":"Q051","category":"Pharmacology","difficulty":"L1","question":"What is the mechanism of action of aspirin?","key_terms":["cox","cyclooxygenase","platelet","thromboxane"],"reference_answer":"Aspirin irreversibly inhibits cyclooxygenase (COX), reducing thromboxane A2 production and thereby inhibiting platelet aggregation."},{"id":"Q052","category":"Pharmacology","difficulty":"L1","question":"What class of drug is metformin, and what is its primary mechanism?","key_terms":["biguanide","hepatic glucose production","insulin sensitivity"],"reference_answer":"Metformin is a biguanide that primarily works by reducing hepatic glucose production and improving peripheral insulin sensitivity."},{"id":"Q053","category":"Pharmacology","difficulty":"L1","question":"What is the mechanism of action of beta-blockers in reducing cardiac workload?","key_terms":["beta receptor","heart rate","contractility","sympathetic"],"reference_answer":"Beta-blockers block beta-adrenergic receptors, reducing heart rate and contractility, which decreases myocardial oxygen demand and cardiac workload."},{"id":"Q054","category":"Pharmacology","difficulty":"L2","question":"What is the mechanism of action of ACE inhibitors and their primary clinical use?","key_terms":["angiotensin","vasodilation","hypertension","heart failure"],"reference_answer":"ACE inhibitors block conversion of angiotensin I to angiotensin II, reducing vasoconstriction and aldosterone release; they are primarily used for hypertension and heart failure."},{"id":"Q055","category":"Pharmacology","difficulty":"L2","question":"Why are NSAIDs contraindicated in patients with acute kidney injury or heart failure?","key_terms":["prostaglandin","renal perfusion","afferent arteriole"],"reference_answer":"NSAIDs inhibit prostaglandin synthesis, which normally maintains afferent arteriolar dilation and renal perfusion; blocking this can worsen renal perfusion and fluid retention in patients with AKI or heart failure."},{"id":"Q056","category":"Pharmacology","difficulty":"L2","question":"What is the mechanism by which loop diuretics cause hypokalemia?","key_terms":["loop of henle","sodium reabsorption","distal tubule","potassium loss"],"reference_answer":"Loop diuretics inhibit sodium reabsorption in the loop of Henle, increasing distal sodium delivery, which promotes sodium-potassium exchange in the distal tubule and increased urinary potassium loss."},{"id":"Q057","category":"Pharmacology","difficulty":"L2","question":"What are the major contraindications to thrombolytic therapy in acute ischemic stroke?","key_terms":["active bleeding","recent surgery","hemorrhage","anticoagulation"],"reference_answer":"Major contraindications include active internal bleeding, recent major surgery or trauma, history of intracranial hemorrhage, and current therapeutic anticoagulation, due to increased bleeding risk."},{"id":"Q058","category":"Pharmacology","difficulty":"L3","question":"Explain the mechanism of warfarin and why vitamin K alone is insufficient for emergent reversal in major bleeding.","key_terms":["vitamin k epoxide reductase","clotting factors","prothrombin complex concentrate"],"reference_answer":"Warfarin inhibits vitamin K epoxide reductase, blocking synthesis of clotting factors II, VII, IX, and X. Vitamin K administration alone takes hours to restore factor synthesis, so emergent major bleeding requires immediate factor replacement with prothrombin complex concentrate or fresh frozen plasma in addition to vitamin K."},{"id":"Q059","category":"Pharmacology","difficulty":"L3","question":"Compare the mechanisms of action of P2Y12 inhibitors and aspirin in preventing arterial thrombosis.","key_terms":["p2y12 receptor","adp","cyclooxygenase","platelet aggregation"],"reference_answer":"Aspirin inhibits cyclooxygenase to reduce thromboxane A2-mediated platelet activation, while P2Y12 inhibitors (e.g., clopidogrel) block the ADP receptor on platelets, preventing ADP-mediated platelet activation; together they provide complementary dual antiplatelet inhibition."},{"id":"Q060","category":"Pharmacology","difficulty":"L3","question":"Explain why combining ACE inhibitors with potassium-sparing diuretics increases the risk of hyperkalemia.","key_terms":["aldosterone","potassium retention","hyperkalemia","renin-angiotensin"],"reference_answer":"ACE inhibitors reduce aldosterone secretion, decreasing renal potassium excretion, while potassium-sparing diuretics directly block potassium excretion in the distal nephron; combining both mechanisms additively increases the risk of hyperkalemia and requires close potassium monitoring."}]'''
eval_set_path = f"{EVAL_DIR}/medilearn_eval_60.json"
if os.path.exists(eval_set_path):
    with open(eval_set_path) as f:
        MEDILEARN_QUESTIONS = json.load(f)
    print(f"✅ Shared eval set loaded from Drive: {eval_set_path}")
else:
    MEDILEARN_QUESTIONS = json.loads(MEDILEARN_EVAL_JSON)
    with open(eval_set_path, "w") as f:
        json.dump(MEDILEARN_QUESTIONS, f, indent=2)
    print(f"ℹ️  Initialized eval set from embedded copy → {eval_set_path}")

assert len(MEDILEARN_QUESTIONS) == 60
from collections import Counter
dc = Counter(q["difficulty"] for q in MEDILEARN_QUESTIONS)
print(f"   L1: {dc['L1']}  |  L2: {dc['L2']}  |  L3: {dc['L3']}  |  Total: 60")

---
## Section 3 — Copy Fine-Tuned Model to Local Colab Disk

The GGUF conversion script reads the model directory byte-by-byte.
Running it directly against the Google Drive mount is slow and can time out —
we copy the model to local `/content/` first (~270 MB for SmolLM-135M).

In [ ]:
import shutil, time

LOCAL_MODEL_DIR = '/content/medilearn_finetuned_hf'  # local copy for fast GGUF conversion
GGUF_F16        = '/content/medilearn_f16.gguf'
GGUF_Q4         = '/content/medilearn_q4_k_m.gguf'
LLAMACPP_DIR    = '/content/llama.cpp'

if os.path.exists(LOCAL_MODEL_DIR):
    shutil.rmtree(LOCAL_MODEL_DIR)

print(f"Copying fine-tuned model from Drive to local disk...")
t0 = time.time()
shutil.copytree(MERGED_MODEL_PATH, LOCAL_MODEL_DIR)
elapsed = time.time() - t0

total_mb = sum(os.path.getsize(os.path.join(LOCAL_MODEL_DIR, f))
               for f in os.listdir(LOCAL_MODEL_DIR) if os.path.isfile(
               os.path.join(LOCAL_MODEL_DIR, f))) / 1024**2

print(f"✅ Copied {total_mb:.0f} MB in {elapsed:.1f}s → {LOCAL_MODEL_DIR}")
print(f"   Files: {sorted(os.listdir(LOCAL_MODEL_DIR))}")

---
## Section 4 — Build llama.cpp Conversion Tools

Same tools regardless of which model we're converting — no changes needed here.

llama.cpp provides two tools we need:
- `convert_hf_to_gguf.py` — Python script that reads HF model files → GGUF
- `llama-quantize` — C++ binary that compresses GGUF F16 → Q4_K_M
- `llama-cli` — C++ binary for running inference (sanity test)

> ⏱️ **~3–4 minutes** to clone and build

In [5]:
import subprocess

# Clone llama.cpp (shallow — latest commit only)
print('Cloning llama.cpp ...')
r = subprocess.run(
    ['git', 'clone', '--depth', '1',
     'https://github.com/ggerganov/llama.cpp', LLAMACPP_DIR],
    capture_output=True, text=True
)
if r.returncode == 0 or 'already exists' in r.stderr:
    print('✅ Cloned')
else:
    print('❌ Clone failed:', r.stderr[-300:])

Cloning llama.cpp ...
✅ Cloned


In [6]:
# Build quantize + cli binaries
print('Configuring build ...')
r = subprocess.run(
    ['cmake', '-B', f'{LLAMACPP_DIR}/build',
     '-S', LLAMACPP_DIR, '-DCMAKE_BUILD_TYPE=Release'],
    capture_output=True, text=True, cwd=LLAMACPP_DIR
)
if r.returncode != 0:
    print('CMake error:', r.stderr[-400:])
else:
    print('✅ CMake configured')

print('Building (3–4 min) ...')
r = subprocess.run(
    ['cmake', '--build', f'{LLAMACPP_DIR}/build',
     '--target', 'llama-quantize', 'llama-cli', '-j4'],
    capture_output=True, text=True
)
if r.returncode != 0:
    print('Build error:', r.stderr[-400:])
else:
    print('✅ Build complete')

# Verify binaries
for b in ['llama-quantize', 'llama-cli']:
    p = f'{LLAMACPP_DIR}/build/bin/{b}'
    print(f'  {"✅" if os.path.exists(p) else "❌"} {p}')

Configuring build ...
✅ CMake configured
Building (3–4 min) ...
✅ Build complete
  ✅ /content/llama.cpp/build/bin/llama-quantize
  ✅ /content/llama.cpp/build/bin/llama-cli


---
## Section 5 — Convert Fine-Tuned MediLearn Model → GGUF F16

`convert_hf_to_gguf.py` reads the HuggingFace model directory and packs
weights + tokenizer + metadata into a single portable GGUF binary.
F16 is lossless — full precision, used as the source for quantization.

> ⏱️ **~2–3 minutes**

In [ ]:
import subprocess, time

print('Converting fine-tuned MediLearn model → GGUF F16 ...')
print(f'  Input : {LOCAL_MODEL_DIR}')
print(f'  Output: {GGUF_F16}')
print()

t0 = time.time()
r = subprocess.run(
    [
        'python', f'{LLAMACPP_DIR}/convert_hf_to_gguf.py',
        LOCAL_MODEL_DIR,
        '--outfile', GGUF_F16,
        '--outtype', 'f16',
    ],
    capture_output=True, text=True
)
elapsed = time.time() - t0

if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])
    raise RuntimeError('GGUF conversion failed')

size_mb = os.path.getsize(GGUF_F16) / 1024**2
print(f'✅ GGUF F16 created in {elapsed:.1f}s  ({size_mb:.0f} MB)')

---
## Section 6 — Quantize to Q4_K_M

**Q4_K_M** = 4-bit weights with K-quant grouping at medium block size.
The community consensus pick for CPU inference — best tradeoff of speed, size, and quality.

| Format | Size | Quality loss | Use when |
|--------|------|-------------|----------|
| F16 | ~3.2 GB | None | Source only — too large |
| Q8_0 | ~1.8 GB | Minimal | RAM allows, quality critical |
| **Q4_K_M** | **~1.1 GB** | **Low** | **← workshop default** |
| Q4_0 | ~0.9 GB | Moderate | Very constrained RAM |

> ⏱️ **~3–4 minutes**

In [ ]:
import subprocess, time

print('Quantizing F16 → Q4_K_M ...')
print(f'  Input : {GGUF_F16}')
print(f'  Output: {GGUF_Q4}')
print()

t0 = time.time()
r = subprocess.run(
    [
        f'{LLAMACPP_DIR}/build/bin/llama-quantize',
        GGUF_F16,
        GGUF_Q4,
        'Q4_K_M',
    ],
    capture_output=True, text=True
)
elapsed = time.time() - t0

if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])
    raise RuntimeError('Quantization failed')

f16_mb = os.path.getsize(GGUF_F16) / 1024**2
q4_mb  = os.path.getsize(GGUF_Q4)  / 1024**2
print(f'✅ Q4_K_M created in {elapsed:.1f}s')
print(f'   F16  : {f16_mb:.0f} MB')
print(f'   Q4_K_M: {q4_mb:.0f} MB  ({q4_mb/f16_mb*100:.0f}% of F16 — {f16_mb/q4_mb:.1f}x smaller)')

---
## Section 7 — MediLearn Evaluation: Full-Precision vs Quantized

This is the workshop's closing measurement. We run:
1. **Full-precision baseline** — load the Module 2 merged model with HuggingFace Transformers
   (same inference path as Modules 3 and 4) and score all 60 questions.
2. **Quantized Q4_K_M** — query the quantized GGUF via `llama-cli` as a subprocess and
   score the same 60 questions.

The comparison answers: *how much accuracy do we trade for the portability and speed of
edge deployment?*

> ⏳ Each model runs 60 questions. On Colab CPU this takes ~20–40 minutes per condition.
> If CPU time is limited, set `FAST_MODE = True` below to use a stratified 20-question sample
> (every 3rd question — spans all categories and difficulty tiers).

**FAST_MODE = False** for full 60-question run (recommended for final demo).
**FAST_MODE = True** for quick check during development.

In [ ]:
import torch, time, subprocess
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

FAST_MODE = False    # ← set True to use 20-question stratified sample

eval_questions = (
    [MEDILEARN_QUESTIONS[i] for i in range(0, 60, 3)] if FAST_MODE
    else MEDILEARN_QUESTIONS
)
n_eval = len(eval_questions)
print(f"Evaluating {n_eval} questions (FAST_MODE={FAST_MODE})")

# Shared embedding model for semantic-relevance scoring
print("Loading embedding model...")
_embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
_ref_embs = _embedder.encode(
    [q["reference_answer"] for q in eval_questions],
    normalize_embeddings=True, show_progress_bar=False
)

def term_coverage_score(response: str, key_terms: list) -> float:
    resp_lower = response.lower()
    hits = sum(1 for t in key_terms if t.lower() in resp_lower)
    return round(hits / len(key_terms), 3) if key_terms else 0.0

def semantic_relevance_score(response: str, ref_emb) -> float:
    resp_emb = _embedder.encode([response], normalize_embeddings=True)[0]
    return round(float(np.dot(resp_emb, ref_emb)), 3)

def score_responses(responses: list, model_label: str) -> pd.DataFrame:
    rows = []
    for q, resp, ref_emb in zip(eval_questions, responses, _ref_embs):
        rows.append({
            "id": q["id"], "category": q["category"], "difficulty": q["difficulty"],
            "question": q["question"], "response": resp,
            "term_coverage": term_coverage_score(resp, q["key_terms"]),
            "semantic_relevance": semantic_relevance_score(resp, ref_emb),
            "model": model_label,
        })
    return pd.DataFrame(rows)

def summarize_medilearn(df):
    by_diff = df.groupby("difficulty")[["term_coverage","semantic_relevance"]].mean().round(3)
    by_diff = by_diff.reindex(["L1","L2","L3"])
    overall = df[["term_coverage","semantic_relevance"]].mean().round(3)
    return by_diff, overall

print("✅ Harness ready")

### 7.1 Condition 1 — Full-Precision Fine-Tuned Model

In [ ]:
# ── Condition 1: Full-Precision Fine-Tuned Model ─────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading full-precision model on {device}...")
fp_tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)
fp_model     = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_DIR,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto", low_cpu_mem_usage=True
)
fp_model.eval()

def gen_full_precision(question: str) -> str:
    msgs = [{"role": "user", "content": question}]
    text = fp_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = fp_tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = fp_model.generate(
            **inputs, max_new_tokens=128, do_sample=True,
            temperature=0.2, top_p=0.9,
            pad_token_id=fp_tokenizer.eos_token_id
        )
    resp = fp_tokenizer.decode(out[0], skip_special_tokens=True)
    return resp.split("assistant\n", 1)[-1].strip() if "assistant\n" in resp else resp.strip()

print(f"Running full-precision eval on {n_eval} questions...")
fp_responses, fp_latencies = [], []
for i, q in enumerate(eval_questions, 1):
    t0 = time.time()
    fp_responses.append(gen_full_precision(q["question"]))
    fp_latencies.append(round(time.time() - t0, 2))
    if i % 10 == 0:
        print(f"  [{i}/{n_eval}]...")

fp_df = score_responses(fp_responses, "Full-Precision (HF Transformers)")
fp_df["latency_sec"] = fp_latencies
by_diff_fp, overall_fp = summarize_medilearn(fp_df)

print(f"\n📊 Full-Precision overall:  term_cov={overall_fp['term_coverage']}  sem_rel={overall_fp['semantic_relevance']}  avg_lat={fp_df['latency_sec'].mean():.1f}s")

# Free GPU memory before loading llama.cpp
import gc; del fp_model
gc.collect()
if device == "cuda": torch.cuda.empty_cache()
print("✅ Full-precision eval done | GPU memory freed")

### 7.2 Condition 2 — Quantized Q4_K_M (llama-cli)

In [ ]:
# ── Condition 2: Quantized Q4_K_M via llama-cli ──────────────────────────────
LLAMA_CLI = f"{LLAMACPP_DIR}/build/bin/llama-cli"

def gen_quantized(question: str, n_predict: int = 128) -> str:
    """Call llama-cli as a subprocess — simulates true edge deployment."""
    prompt = f"<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"
    r = subprocess.run(
        [LLAMA_CLI, "--model", GGUF_Q4, "--prompt", prompt,
         "--n-predict", str(n_predict), "--threads", "4",
         "--temp", "0.2", "--top-p", "0.9", "--no-display-prompt", "--log-disable"],
        capture_output=True, text=True, timeout=120
    )
    out = r.stdout.strip()
    # llama-cli sometimes appends metadata lines after output; strip them
    lines = [l for l in out.split("\n") if not l.startswith("[") and not l.startswith("llama")]
    return "\n".join(lines).strip()

print(f"Running Q4_K_M quantized eval on {n_eval} questions via llama-cli...")
q4_responses, q4_latencies = [], []
for i, q in enumerate(eval_questions, 1):
    t0 = time.time()
    q4_responses.append(gen_quantized(q["question"]))
    q4_latencies.append(round(time.time() - t0, 2))
    if i % 10 == 0:
        print(f"  [{i}/{n_eval}]...")

q4_df = score_responses(q4_responses, "Quantized Q4_K_M (llama-cli)")
q4_df["latency_sec"] = q4_latencies
by_diff_q4, overall_q4 = summarize_medilearn(q4_df)

print(f"\n📊 Quantized Q4_K_M overall:  term_cov={overall_q4['term_coverage']}  sem_rel={overall_q4['semantic_relevance']}  avg_lat={q4_df['latency_sec'].mean():.1f}s")
print("✅ Quantized eval done")

### 7.3 Comparison Chart & Summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle(
    f"Module 6: Full-Precision vs Q4_K_M Quantized Fine-Tuned MediLearn Model\n"
    f"({'60' if not FAST_MODE else '20'}-question shared eval set)",
    fontsize=12, fontweight="bold"
)

tiers, tier_labels = ["L1","L2","L3"], ["L1\nBasic recall","L2\nClinical scenario","L3\nAdvanced reasoning"]
x, w = np.arange(3), 0.35

ax = axes[0]
ax.bar(x-w/2, by_diff_fp["term_coverage"],  w, label="Full-Precision", color="#3498db", alpha=0.85)
ax.bar(x+w/2, by_diff_q4["term_coverage"],  w, label="Q4_K_M Quantized", color="#e67e22", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(tier_labels, fontsize=8)
ax.set_ylim(0,1); ax.set_ylabel("Term coverage"); ax.set_title("Accuracy by tier"); ax.legend(fontsize=8)

ax2 = axes[1]
ax2.bar(x-w/2, by_diff_fp["semantic_relevance"], w, label="Full-Precision", color="#3498db", alpha=0.85)
ax2.bar(x+w/2, by_diff_q4["semantic_relevance"], w, label="Q4_K_M Quantized", color="#e67e22", alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels(tier_labels, fontsize=8)
ax2.set_ylim(0,1); ax2.set_ylabel("Semantic relevance"); ax2.set_title("Relevance by tier"); ax2.legend(fontsize=8)

ax3 = axes[2]
models   = ["Full-Precision", "Q4_K_M"]
latencies = [fp_df["latency_sec"].mean(), q4_df["latency_sec"].mean()]
bars = ax3.bar(models, latencies, color=["#3498db","#e67e22"], alpha=0.85)
ax3.set_ylabel("Avg latency (sec/question)")
ax3.set_title("Latency comparison (Colab CPU)")
for bar, v in zip(bars, latencies):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
             f"{v:.1f}s", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("/tmp/medilearn_m6_eval.png", dpi=150, bbox_inches="tight")
plt.show()

acc_drop = round(overall_fp["term_coverage"] - overall_q4["term_coverage"], 3)
lat_gain = round((fp_df["latency_sec"].mean() - q4_df["latency_sec"].mean()) / fp_df["latency_sec"].mean() * 100, 1)
print(f"\nAccuracy drop (full-precision → Q4_K_M): {acc_drop:+.3f} term coverage")
print(f"Latency change: {lat_gain:+.1f}%  (negative = quantized is faster)")
print(f"\nThis is the accuracy–portability trade-off of edge deployment.")
print(f"For a medical education assistant, an acceptable threshold is typically < 0.05 term-coverage drop.")

---
## Section 8 — Download to Your Laptop

The Q4_K_M file size depends on the model — SmolLM-135M Q4_K_M is ~100 MB; SmolLM-1.7B would be ~1.1 GB. Your fine-tuned 135M model will be the smaller size.

| Connection | Estimated time |
|---|---|
| 100 Mbps | ~2 min |
| 50 Mbps | ~3–4 min |
| 20 Mbps | ~8–10 min |

> 💡 **While the download runs** — open a terminal on your laptop and start
> installing llama.cpp (see the next section). Both steps take about the same time.

In [13]:
from google.colab import files

size_mb = os.path.getsize(GGUF_Q4) / 1024**2
print(f'File : {GGUF_Q4}')
print(f'Size : {size_mb:.0f} MB  ({size_mb/1024:.2f} GB)')
print()
print('Suggested save location on your laptop:')
print('  macOS/Linux : ~/models/smollm_1.7b_q4_k_m.gguf')
print('  Windows     : C:\\Users\\YOU\\models\\smollm_1.7b_q4_k_m.gguf')
print()
files.download(GGUF_Q4)

File : /content/smollm_135M_q4_k_m.gguf
Size : 101 MB  (0.10 GB)

Suggested save location on your laptop:
  macOS/Linux : ~/models/smollm_1.7b_q4_k_m.gguf
  Windows     : C:\Users\YOU\models\smollm_1.7b_q4_k_m.gguf



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Section 9 — Laptop Setup (run in your terminal)

While the download is running, set up llama.cpp on your laptop.

### Install llama.cpp

**macOS**
```bash
brew install llama.cpp
```

**Linux**
```bash
git clone --depth 1 https://github.com/ggerganov/llama.cpp
cd llama.cpp
cmake -B build -DCMAKE_BUILD_TYPE=Release
cmake --build build -j$(nproc)
sudo cmake --install build --prefix /usr/local
```

**Windows**
```bash
winget install ggerganov.llama.cpp
# or download pre-built zip from:
# https://github.com/ggerganov/llama.cpp/releases
```

---

### First inference test
```bash
llama-cli \
  --model ~/models/smollm_1.7b_q4_k_m.gguf \
  --prompt "<|im_start|>user\nTranslate to French: Good morning.<|im_end|>\n<|im_start|>assistant\n" \
  --n-predict 50 \
  --threads 4 \
  --no-display-prompt
```

---

### Launch the local API server (for the audience demo)
```bash
llama-server \
  --model ~/models/smollm_1.7b_q4_k_m.gguf \
  --host 0.0.0.0 \
  --port 8080 \
  --threads 4 \
  --ctx-size 2048
```

When you see `HTTP server listening` — the server is ready.
Open `http://localhost:8080` to confirm, then share your local IP with the audience.

```bash
# Find your local IP
# macOS/Linux:
ifconfig | grep 'inet ' | grep -v 127.0.0.1
# Windows:
ipconfig | findstr IPv4
```

---
## Section 10 — Benchmarking Worksheet

Run these commands on your laptop with the downloaded MediLearn Q4_K_M model.
**Use the same medical prompt each time** — this lets you compare thread counts consistently,
and lets you compare your laptop TPS to the Colab TPS from Section 7.

```bash
MODEL=~/models/medilearn_q4_k_m.gguf
PROMPT="<|im_start|>user\nWhat are the first-line treatments for bacterial meningitis in adults?<|im_end|>\n<|im_start|>assistant\n"

llama-cli --model $MODEL --prompt "$PROMPT" --n-predict 128 --threads 2 --no-display-prompt
llama-cli --model $MODEL --prompt "$PROMPT" --n-predict 128 --threads 4 --no-display-prompt
llama-cli --model $MODEL --prompt "$PROMPT" --n-predict 128 --threads 8 --no-display-prompt
```

| Config | Load time | TPS (eval rate) | RAM used | Response quality |
|--------|-----------|-----------------|----------|-----------------|
| Q4_K_M · 2 threads | &nbsp; | &nbsp; | &nbsp; | &nbsp; |
| Q4_K_M · 4 threads | &nbsp; | &nbsp; | &nbsp; | &nbsp; |
| Q4_K_M · 8 threads | &nbsp; | &nbsp; | &nbsp; | &nbsp; |

**Discussion questions:**
1. At what thread count did TPS stop improving? Why?
2. How does laptop TPS compare to the Colab TPS from Section 7?
3. How much accuracy (term coverage) did you lose vs the full-precision result in Section 7?
4. Is that trade-off acceptable for a medical education app that must run offline on a laptop?
5. What would you do differently for a production edge deployment on a phone or microcontroller?


In [ ]:
# ── Save results to Drive ─────────────────────────────────────────────────────
fp_df.to_json(os.path.join(DRIVE_RESULTS_DIR, "module6_fullprecision.json"), orient="records", indent=2)
q4_df.to_json(os.path.join(DRIVE_RESULTS_DIR, "module6_q4km.json"), orient="records", indent=2)

with open(os.path.join(DRIVE_RESULTS_DIR, "module6_summary.json"), "w") as f:
    json.dump({
        "eval_mode": "FAST_MODE" if FAST_MODE else "FULL_60",
        "n_questions": n_eval,
        "full_precision": {**overall_fp.to_dict(), "avg_latency_sec": round(fp_df["latency_sec"].mean(), 2)},
        "q4_k_m":         {**overall_q4.to_dict(), "avg_latency_sec": round(q4_df["latency_sec"].mean(), 2)},
        "accuracy_drop":  acc_drop,
        "latency_change_pct": lat_gain,
    }, f, indent=2)

print("✅ Results saved → Drive/eval/results/module6_*.json")

# ── Workshop closing summary ───────────────────────────────────────────────────
print()
print("=" * 70)
print("MEDILEARN MODULE 6 COMPLETE — Workshop Summary")
print("=" * 70)
print()
print("Model pipeline built across 5 modules:")
print("  Module 1 → Baseline scores (un-fine-tuned SmolLM-135M on 60 Q set)")
print("  Module 2 → Fine-tuned scores + merged model saved to Drive")
print("  Module 3 → Fine-tuned SLM + RAG scores")
print("  Module 4 → Passive RAG vs Agentic RAG scores")
print("  Module 6 → Full-precision vs Q4_K_M quantized scores  ← just completed")
print()
print("Module 6 accuracy/latency trade-off:")
print(f"  Full-precision term coverage : {overall_fp['term_coverage']:.3f}")
print(f"  Q4_K_M term coverage         : {overall_q4['term_coverage']:.3f}  (Δ {acc_drop:+.3f})")
print(f"  Latency change (Colab)       : {lat_gain:+.1f}%")
print()
print("Model is now:")
print("  🔒 Completely local — no API key, no cloud")
print("  ⚡ CPU-only — runs on any laptop")
print("  📦 Self-contained — one GGUF file, no Python dependencies")
print("  🌐 Serveable — llama-server exposes an OpenAI-compatible REST API")
print()
print("All 60-question eval results saved to:")
print(f"  {DRIVE_RESULTS_DIR}/")
print("  module1_baseline.json · module2_baseline.json · module2_finetuned.json")
print("  module3_slm.json · module3_slm_rag.json · module4_passive_rag_60q.json")
print("  module4_agentic_rag_60q.json · module6_fullprecision.json · module6_q4km.json")
print("=" * 70)

---
## ✅ Module 6 Complete

| Step | What happened |
|------|---------------|
| Loaded fine-tuned MediLearn model from Drive | Same model Module 2 produced |
| Copied to local Colab disk | For fast GGUF conversion |
| Converted to GGUF F16 | Using llama.cpp `convert_hf_to_gguf.py` |
| Quantized to Q4_K_M | 3–4x smaller, minimal quality loss |
| Evaluated full-precision vs quantized | Same 60-question MediLearn set as every other module |
| Downloaded to laptop | Ready for local inference |

**The complete workshop accuracy progression (all 60 questions, term coverage):**

| Module | Condition | What it proves |
|--------|-----------|----------------|
| 1 | Base SmolLM-135M (no fine-tuning) | Starting point |
| 2 | Fine-tuned on medical flashcards | Fine-tuning adds domain knowledge |
| 3 | Fine-tuned SLM + RAG | RAG fills knowledge-base gaps |
| 4 | Agentic RAG | Active reasoning beats passive retrieval on L3 questions |
| 6 | Q4_K_M quantized | Edge deployment costs < 0.05 term-coverage drop |

---
*Module 6 · Edge AI & TinyML · Mastering Small Language Models for Real-World AI Systems*
